# Underwater Object Detection v2

**Primary model:** YOLOv9c on URPC 2019 (5 marine classes)  
**Optional baseline:** YOLO11s

### Setup (Kaggle)
1. Attach dataset: **`urpc2019-640-15pct`** ([create with `prepare_urpc640.py`](../prepare_urpc640.py))
2. Settings → **GPU T4 x2**, Internet **On**, Restart Session
3. Set `RUN_SMOKE_TEST = False` for full training (or `True` for 1-epoch CPU smoke test)
4. **Save Version** after run to download weights and predictions

**Kaggle notebook:** https://www.kaggle.com/code/salonichippa/underwater-yolo


In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml

In [ ]:
import os
import glob
import gc
import yaml
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
print('Ultralytics YOLO ready')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Configuration

CLASSES = ['echinus', 'starfish', 'holothurian', 'scallop', 'waterweeds']

RUN_SMOKE_TEST = False   # True = CPU smoke test (1 epoch, 10% data)
RUN_BASELINE = False     # True = also train YOLO11s for comparison

IMGSZ = 640
SEED = 88

if RUN_SMOKE_TEST:
    PRIMARY_MODEL = 'yolov9c.pt'
    EPOCHS = 1
    BATCH_SIZE = 2
    FRACTION = 0.10
    DEVICE = 'cpu'
    RUN_NAME = 'smoke_yolov9c'
else:
    PRIMARY_MODEL = 'yolov9c.pt'
    EPOCHS = 80
    BATCH_SIZE = 4          # use 2 if OOM
    FRACTION = 1.0
    RUN_NAME = 'yolov9c_urpc640_15pct'
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable GPU T4 and Restart Session before full training.')
    DEVICE = 0

BASELINE_MODEL = 'yolo11s.pt'
BASELINE_RUN_NAME = 'yolo11s_urpc640_15pct_baseline'

OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
DATA_YAML = os.path.join(OUTPUT_DIR, 'data.yaml')


def _is_valid_dataset(root: str) -> bool:
    train_img = os.path.join(root, 'train', 'images')
    return os.path.isdir(train_img) and len(os.listdir(train_img)) > 0


def find_dataset_dir():
    candidates = []
    if os.path.isdir('/kaggle/input'):
        for name in sorted(os.listdir('/kaggle/input')):
            candidates.append(os.path.join('/kaggle/input', name))
    candidates.extend(['./data/urpc2019-640-15pct', '../data/urpc2019-640-15pct'])
    for root in candidates:
        if _is_valid_dataset(root):
            return root
        if os.path.isdir(root):
            for sub in sorted(os.listdir(root)):
                subpath = os.path.join(root, sub)
                if _is_valid_dataset(subpath):
                    return subpath
    return None


DATASET_DIR = find_dataset_dir()
if DATASET_DIR is None:
    raise FileNotFoundError('Attach urpc2019-640-15pct dataset via Add Data.')

print('Mode:', 'SMOKE TEST' if RUN_SMOKE_TEST else 'FULL TRAINING')
print('Dataset:', DATASET_DIR)
print('Model:', PRIMARY_MODEL, '| Epochs:', EPOCHS, '| Batch:', BATCH_SIZE, '| Device:', DEVICE)


In [ ]:
# Write data.yaml pointing to attached dataset
dataset_yaml = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(CLASSES),
    'names': CLASSES,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)

print('data.yaml:')
print(open(DATA_YAML).read())

In [ ]:
# Verify dataset structure and class distribution
def count_dataset(base_dir):
    rows = []
    for split in ['train', 'val', 'test']:
        img_dir = os.path.join(base_dir, split, 'images')
        lbl_dir = os.path.join(base_dir, split, 'labels')
        n_img = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
        class_counts = {c: 0 for c in CLASSES}
        if os.path.isdir(lbl_dir):
            for lf in os.listdir(lbl_dir):
                with open(os.path.join(lbl_dir, lf)) as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            cid = int(float(parts[0]))
                            if 0 <= cid < len(CLASSES):
                                class_counts[CLASSES[cid]] += 1
        rows.append({'split': split, 'images': n_img, **class_counts})
    return pd.DataFrame(rows)

stats_df = count_dataset(DATASET_DIR)
display(stats_df)

# Sanity check: sample image size
sample = glob.glob(os.path.join(DATASET_DIR, 'train', 'images', '*'))[0]
with Image.open(sample) as im:
    print(f'Sample: {os.path.basename(sample)} → {im.size}')

In [ ]:
# Visualize one training image with labels
def plot_sample(img_path, lbl_path):
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img)
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cid, xc, yc, bw, bh = parts
                cid = int(float(cid))
                xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                rect_w, rect_h = bw * w, bh * h
                ax.add_patch(plt.Rectangle((x1, y1), rect_w, rect_h,
                             fill=False, edgecolor='lime', linewidth=2))
                ax.text(x1, y1 - 4, CLASSES[cid], color='lime', fontsize=8,
                        bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax.set_title(os.path.basename(img_path))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

s_img = glob.glob(os.path.join(DATASET_DIR, 'train', 'images', '*'))[0]
s_lbl = os.path.join(DATASET_DIR, 'train', 'labels', Path(s_img).stem + '.txt')
plot_sample(s_img, s_lbl)

In [ ]:
%%time
# ── Train YOLOv9c (primary) ───────────────────────────────────────────────

model = YOLO(PRIMARY_MODEL)

results = model.train(
    data=DATA_YAML,
    task='detect',
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    fraction=FRACTION,
    optimizer='AdamW',
    lr0=1e-4,
    lrf=0.01,
    cos_lr=True,                # NEW — smoother LR decay
    weight_decay=0.0005,
    patience=15,
    name=RUN_NAME,
    seed=SEED,
    device=DEVICE,
    amp=True,
    cache=False,
    workers=2,
    exist_ok=True,
    verbose=True,
    # Underwater-friendly augmentation
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.1,             # helps rare classes (scallop, echinus)
    close_mosaic=10,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Training complete. Run name:', RUN_NAME)

In [ ]:
# ── Evaluate on test split ────────────────────────────────────────────────

best_weights = os.path.join('runs', 'detect', RUN_NAME, 'weights', 'best.pt')
assert os.path.exists(best_weights), f'Weights not found: {best_weights}'

best_model = YOLO(best_weights)
metrics = best_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, device=DEVICE)

print('\n=== Test metrics (YOLOv9c — 15% URPC) ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

In [ ]:
# Training curves
results_csv = os.path.join('runs', 'detect', RUN_NAME, 'results.csv')
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    display(df.tail())
    df.to_csv(os.path.join(OUTPUT_DIR, f'{RUN_NAME}_results.csv'), index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if 'train/box_loss' in df.columns:
        axes[0].plot(df['train/box_loss'], label='train box loss')
        axes[0].plot(df['val/box_loss'], label='val box loss')
        axes[0].legend(); axes[0].set_title('Box Loss')
    if 'metrics/mAP50(B)' in df.columns:
        axes[1].plot(df['metrics/mAP50(B)'], label='mAP50')
        axes[1].legend(); axes[1].set_title('mAP@0.5')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'{RUN_NAME}_curves.png'), dpi=120)
    plt.show()

In [ ]:
# Predict on test images and save visualizations
test_imgs = glob.glob(os.path.join(DATASET_DIR, 'test', 'images', '*'))[:8]
pred_dir = os.path.join(OUTPUT_DIR, f'{RUN_NAME}_predictions')

preds = best_model.predict(
    source=test_imgs,
    imgsz=IMGSZ,
    conf=0.25,
    save=True,
    project=OUTPUT_DIR,
    name=f'{RUN_NAME}_predictions',
    exist_ok=True,
    device=DEVICE,
)

saved = glob.glob(os.path.join(pred_dir, '*'))
print(f'Saved {len(saved)} prediction images to {pred_dir}')

In [ ]:
# Copy best weights to Kaggle output (persists when you Save Version)
import shutil

out_weights = os.path.join(OUTPUT_DIR, f'{RUN_NAME}_best.pt')
shutil.copy(best_weights, out_weights)
print('Saved:', out_weights)

---
## Optional: YOLO11s baseline

Run this section **only after** the YOLOv9c full run succeeds.  
Set `RUN_BASELINE = True` in the config cell and re-run from here.

In [ ]:
if not RUN_BASELINE:
    print('Skipping YOLO11s baseline. Set RUN_BASELINE = True to enable.')
else:
    assert not RUN_SMOKE_TEST, 'Disable smoke test before running baseline.'

    baseline = YOLO(BASELINE_MODEL)
    baseline.train(
        data=DATA_YAML,
        imgsz=IMGSZ,
        epochs=50,
        batch=BATCH_SIZE,
        optimizer='AdamW',
        lr0=1e-4,
        name=BASELINE_RUN_NAME,
        seed=SEED,
        device=0,
        amp=True,
        cache=False,
        workers=2,
        exist_ok=True,
    )

    bl_weights = os.path.join('runs', 'detect', BASELINE_RUN_NAME, 'weights', 'best.pt')
    bl_model = YOLO(bl_weights)
    bl_metrics = bl_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, device=0)

    print('\n=== Test metrics (YOLO11s baseline) ===')
    print(f'mAP@0.5:      {bl_metrics.box.map50:.4f}')
    print(f'mAP@0.5:0.95: {bl_metrics.box.map:.4f}')
    print(f'Precision:    {bl_metrics.box.mp:.4f}')
    print(f'Recall:       {bl_metrics.box.mr:.4f}')

---
### After smoke test passes

1. Set `RUN_SMOKE_TEST = False` in the config cell
2. Enable **GPU** in notebook settings
3. Re-run from config → train → evaluate
4. **Save Version** with output to keep `yolov9c_urpc640_best.pt` and prediction images